# 중기부 점검 후보군과 복수 시나리오 순위 안정성

## TL;DR

- 프로그램-연도-회계유형 66행 중 공동분석 58행, 설명 가능한 점검 후보 45행입니다.
- 결측을 0점 처리하지 않고 순위를 비교할 수 있는 후보는 38행입니다.
- 4개 시나리오의 순위상관은 0.66~0.91입니다.
- 상위 5개는 모든 시나리오에서 같고, 상위 10개는 공통 8개·합집합 14개입니다.
- 따라서 최상위 후보는 비교적 안정적이지만 6~10위권은 가중치 선택에 민감합니다.

## 분석 정의

분석 단위는 `부처 × 프로그램 × 회계연도 × 회계유형`입니다. 성과미달 비중, 집행관리 신호, 성과-예산변화 불일치, 재정영향도를 사용합니다. 균등가중·성과중심·집행중심·재정영향 보정 시나리오를 비교하며, 결과는 최종 정책 우선순위가 아닌 탐색용 점검 순위입니다.

In [ ]:
from pathlib import Path

from IPython.display import Image, display

from analytics.mss_priority_scenario_analysis import (
    PriorityScenarioPaths,
    run_priority_scenario_analysis,
)

root = Path.cwd()
result = run_priority_scenario_analysis(
    PriorityScenarioPaths.from_root(root), overwrite=True
)
result.summary

## 후보군 구성

데이터 검증 대상은 순위에 넣지 않고 별도 큐로 유지합니다. 회계조정·프로그램 구조 신호만 있는 행도 맥락 검토 대상으로 보존하되, 사업규모만으로는 후보를 만들지 않습니다.

In [ ]:
candidate_counts = (
    result.candidates.groupby(
        ["priority_tier", "review_candidate", "scenario_ranking_eligible"],
        dropna=False,
    )
    .size()
    .rename("row_count")
    .reset_index()
)
candidate_counts

## 순위 안정성

점은 네 시나리오 평균순위, 선은 최상·최하 순위 범위입니다. 선이 짧을수록 가중치 변경에도 순위가 안정적입니다.

In [ ]:
columns = [
    "exploratory_consensus_order",
    "fiscal_year",
    "performance_program_name",
    "account_type",
    "priority_reason",
    "mean_scenario_rank",
    "best_scenario_rank",
    "worst_scenario_rank",
    "scenario_rank_range",
    "top_5_scenario_count",
    "top_10_scenario_count",
]
display(result.stability[columns].head(15))
display(Image(filename=str(result.figure_paths[0])))

In [ ]:
spearman_matrix = result.spearman.pivot(
    index="scenario_left",
    columns="scenario_right",
    values="spearman_rank_correlation",
)
display(spearman_matrix)
display(result.top_k_overlap.query("comparison_type == 'ALL_SCENARIOS'"))
display(Image(filename=str(result.figure_paths[1])))

## 해석과 남은 한계

상위 5개는 네 시나리오에서 유지되므로 우선 원문 검토 대상으로 방어하기 쉽습니다. 반면 상위 10개 경계는 시나리오에 따라 바뀌므로 단일 가중치 순위를 확정하면 안 됩니다. 현재 수기표에 성과지표 유형과 자율평가 의견이 없어 구성요소에 포함하지 않았고, 결측을 0점 처리하거나 나머지 가중치로 재배분하지 않았습니다.